<a href="https://colab.research.google.com/github/acarsondave/tiktok-scraper/blob/main/tiktok_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from selenium import webdriver
import time
from datetime import datetime
import pandas
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome('chromedriver',options=options)
driver1 = webdriver.Chrome('chromedriver',options=options)
driver2 = webdriver.Chrome('chromedriver',options=options)
driver.get("https://www.tiktok.com/music/Because-of-You-6896223884782077953?_d=secCgsIARCbDRgBIAIoARI%2BCjx4jnuX7AefMNdouEZOo9gWbO2w5RDaJ1y7trVnIWTK0h21hJDaKqbswIqWt3g3iKHs0bHwT7K7rfEJdSAaAA%3D%3D&language=en&sec_user_id=MS4wLjABAAAAnbamQ47M0ESG8Bl6-PuMhVXof-dmWv1NXkh7NvgaQwehfBpFPY9CwbdsVVHuG43z&share_link_id=2D160053-4B13-4F16-B466-700E043190CF&share_music_id=6896223884782077953&tt_from=copy&u_code=def8cliggdfga9&user_id=6873924327805649926&utm_campaign=client_share&utm_medium=ios&utm_source=copy&source=h5_m")
driver.implicitly_wait(10) # seconds


In [ ]:
soup = BeautifulSoup(driver.page_source,'html.parser')
head = soup.find('div',{'class':'share-title-container'})
song_name = head.find('h1',{'class':'share-title'}).text
song_author = head.find('h2',{'class':'share-sub-title'}).text
videos = head.find('h2',{'class':'share-sub-title-thin'}).text
print("Song name - %s\nSong Author - %s\nVideo count - %s" %(song_name,song_author,videos))
if 'K' in videos:
  if '.' in videos:
    videos = videos.split('.')[0]
    video_c = videos + "000"
    print(video_c)
  else:
    videos = videos.split("K")[0]
    video_c = videos + "000"
    print(video_c)
else:
  videos = videos.split('v')[0]
  video_c = videos.strip()
  print(video_c)
if 'M' in videos:
  if '.' in videos:
    videos = videos.split('.')[0]
    video_c = videos + "000000"
    print(video_c)
  else:
    videos = videos.split("M")[0]
    video_c = videos + "000000"
    print(video_c)

Song name - Because of You
Song Author - Future & Lil Uzi Vert
Video count - 63 videos
63


In [ ]:
wait = WebDriverWait(driver2, 10)
def getvideoinfo(d,l,url,item):
  soup = BeautifulSoup(driver1.page_source,'html.parser')
  #wait.until(EC.presence_of_element_located((By.CLASS_NAME, "user-username")))
  user_username = driver1.find_element_by_class_name("author-uniqueId").text
  soup1 = BeautifulSoup(driver.page_source,'html.parser')
  music_author = soup1.find("h2",{"class":"share-sub-title"}).text
  getvideoinfo.ma = music_author
  try:
    video_info = driver1.find_element_by_class_name("tt-video-meta-caption").text
  except:
    video_info = "no description for this user"
  video_likes = soup.find("strong",{"title":"like"}).text
  try:
    video_share = soup.find("strong",{"title":"share"}).text
  except:
    video_share = 0
  video_comments = soup.find("strong",{"title":"comment"}).text
  user_nickname,date_posted = driver1.find_element_by_class_name("author-nickname").text.split('·')
  user_nickname = user_nickname.strip()
  date_posted = date_posted.strip()
  music_id = driver.current_url
  music_id = music_id.split("?")[0]
  music_id = music_id.split("music")[1]
  music_id = re.sub("[A-Za-z]|-","",music_id)
  getvideoinfo.mi = music_id
  video_id = driver1.current_url
  video_id = video_id.split("?")[0]
  video_id = video_id.split("video/")[1]
  video_url = driver1.current_url
  print(url[item]["href"])
  nurl = url[item]["href"].split("/video")[0]
  driver2.get(nurl)
  try:
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "count-infos")))
    soup = BeautifulSoup(driver2.page_source,'html.parser')
    following = soup.find("strong",{"title":"Following"}).text
    followers = soup.find("strong",{"title":"Followers"}).text
    likes = soup.find("strong",{"title":"Likes"}).text
    profile_desc = soup.find("h2",{"class":"share-desc"}).text
    profile_url = driver2.current_url
    try:
      is_verified = soup.find("h2",{"class":"verified"}).text
      is_verified = "True"
    except AttributeError as e:
      is_verified = "False"
    d["User_Followers"] = followers
    d["User_Following"] = following
    d["User_Hearts"] = likes
    d["Profile_Description"] = profile_desc
    d["Is_Verified"] = is_verified
    d["Profile_url"] = profile_url
    old_url = 0
    vid_count = '0'
    play_c = ''
    while True:
      soup = BeautifulSoup(driver2.page_source,'html.parser')
      url1 = soup.find_all("a",{"class":"video-feed-item-wrapper"})
      print(len(url1))
      if old_url == len(url1):
        print("done")
        break

      if vid_count == '0':
        for item1 in range(len(url1)):
          if url1[item1]["href"] == url[item]["href"]:
            play_c = driver2.find_element_by_class_name("video-count").text
            print("Found video, " + play_c)

      vid_count = play_c
      driver2.execute_script("window.scrollTo(0, document.body.scrollHeight);")
      time.sleep(1)
      old_url = len(url1)

  except TimeoutException :
    print("invalid account")

  '''try:
    driver.find_element_by_class_name("close").click()
  except:
    print("already closed")'''
  #driver.get('https://www.tiktok.com/music/Because-of-You-6896223884782077953?_d=secCgsIARCbDRgBIAIoARI%2BCjx4jnuX7AefMNdouEZOo9gWbO2w5RDaJ1y7trVnIWTK0h21hJDaKqbswIqWt3g3iKHs0bHwT7K7rfEJdSAaAA%3D%3D&language=en&sec_user_id=MS4wLjABAAAAnbamQ47M0ESG8Bl6-PuMhVXof-dmWv1NXkh7NvgaQwehfBpFPY9CwbdsVVHuG43z&share_link_id=2D160053-4B13-4F16-B466-700E043190CF&share_music_id=6896223884782077953&tt_from=copy&u_code=def8cliggdfga9&user_id=6873924327805649926&utm_campaign=client_share&utm_medium=ios&utm_source=copy&source=h5_m')
  d["Username"] = user_username
  d["User_Nickname"] = user_nickname
  d["Video_ID"] = video_id
  d["Description"] = video_info
  d["Video_Hearts"] = video_likes
  d["Video_Shares"] = video_share
  d["Video_comments"] = video_comments
  d["Date_Posted"] = date_posted
  d["Video_Url"] = video_url
  d["User_video_Uploads"] = old_url
  d["Video plays"] = vid_count
  l.append(d)
  getvideoinfo.ci = " User's Username - %s \n Video's description - %s \n Video's Likes and Comments - %s likes  , %s comments \n Video's url - %s \n" %(user_username,video_info,video_likes,video_comments,video_url)

def parse():
  soup = BeautifulSoup(driver.page_source,'html.parser')
  feed = soup.find_all('div',{'class':'video-feed-item'})
  print(len(feed))
  l = []
  for item in range(int(video_c)):
    print("current round - " + str(item + 1))
    d = {}
    soup = BeautifulSoup(driver.page_source,'html.parser')
    url = soup.find_all("a",{"class":"video-feed-item-wrapper"})
    try:
      element = driver.find_elements_by_class_name("video-feed-item")[item]
    except IndexError:
      break
    driver.execute_script("arguments[0].scrollIntoView();", element)
    driver1.get(url[item]["href"])
    getvideoinfo(d,l,url,item)
  d["Music_Author"] = getvideoinfo.ma
  d["Music_ID"] = getvideoinfo.mi
  d["Total_Videos"] = item
  df = pandas.DataFrame(l)
  df.to_csv("data.csv",mode='w',header=True)
  print(datetime.now() - startTime)

startTime = datetime.now()

parse()

56
current round - 1
https://www.tiktok.com/@lr9ine/video/6898143376848211206
36
Found video, 1624
66
93
93
done
current round - 2
https://www.tiktok.com/@gta_photographers/video/6896436827377519877
36
Found video, 440
45
45
done
current round - 3
https://www.tiktok.com/@sweqrvert/video/6896566501927619842
21
Found video, 1202
39
51
69
83
97
121
141
161
179
200
218
248
278
307
336
366
396
407
407
done
current round - 4
https://www.tiktok.com/@newmusicdaily4you/video/6897695496601750790
30
Found video, 1
30
done
current round - 5
https://www.tiktok.com/@braydentomisthebomb/video/6899896993951288582
13
Found video, 166
13
done
current round - 6
https://www.tiktok.com/@peytonmckenzie__/video/6899509499971865862
14
Found video, 1202
14
done
current round - 7
https://www.tiktok.com/@sidelinehighlights/video/6899217068713921797
7
Found video, 2
7
done
current round - 8
https://www.tiktok.com/@levicroswell10/video/6897828199443582214
20
Found video, 134
25
28
28
done
current round - 9
https:/

NameError: ignored

In [ ]:
driver.quit()
driver1.quit()
driver2.quit()

NameError: ignored

In [ ]:
gcloud compute os-login ssh-keys add \ --key-file="C:\Users\Scorpion\Documents\Upwork\Python\tiktok scraper" \ --ttl=0

calc(400px + (100vw - 768px) / 1152 * 100)


ValueError: ignored

In [ ]:
startTime = datetime.now()
for item in range(int(video_c)):
  soup = BeautifulSoup(driver.page_source,'html.parser')
  url = soup.find_all("a",{"class":"video-feed-item-wrapper"})
  try:
    element = driver.find_elements_by_class_name("video-feed-item")[item]
  except IndexError:
    break
  driver.execute_script("arguments[0].scrollIntoView();", element)
  driver1.get(url[item]["href"])
  nurl = url[item]["href"].split("/video")[0]
  driver2.get(nurl)
  print(driver1.title)
  desc = driver2.find_element_by_class_name("share-desc").text
  print(desc)
  print("%s , next..." %(item))

print(datetime.now() - startTime)


5 cuts,Two Books! 🔨 #roofing #fyp
Instagram- LR9INE Roofing 🔨
0 , next...
#rockstargames #carmeet #gta5 #donk #lowrider #lowriderlifemagazine #fyp #foryoupage #foryou #gta5online #hydraulics
Love all consoles but Xbox has my heart follow for more 

Gt:xSADA BABYx 

❤️
1 , next...
#fyp #xyzbca #viral 🙆🏿‍♂️
Ceo of Waa
3k family
I luv Trollin ppl
If you’re easily offended keep scrollin
2 , next...
#fyp| | with Music Because of You - Future & Lil Uzi Vert
Give a follow if you are reading this🤷‍♂️🤷‍♂️
3 , next...
eek| | with Music Because of You - Future & Lil Uzi Vert
these are basically videos i would put in my draft 🤌🏼
main:@braydentomberlin
4 , next...
dancing off our thanksgiving dinner #fyp #foryou #thankful #bestie @hannahcotton9
Nashville - Chattanooga 
insta: peytonmckenzie_
snap: all_beyoutiful
5 , next...
#alexovechkin #nhl #hockey #fyp #xyzbca #foryou
All the best highlights in sports and culture.
6 , next...
#fyp| | with Music Because of You - Future & Lil Uzi Vert
snap// levi.

In [ ]:
driver2.get("https://www.tiktok.com/@dennis_main?lang=en")

old_url = 0
vid_count = '0'
play_c = ''
while True:
  soup = BeautifulSoup(driver2.page_source,'html.parser')
  url1 = soup.find_all("a",{"class":"video-feed-item-wrapper"})
  print(len(url1))
  if old_url == len(url1):
    print("done")
    break

  if vid_count == '0':
    for item in range(len(url1)):
      if url1[item]["href"] == "https://www.tiktok.com/@dennis_main/video/6899503099199753474":
        play_c = driver2.find_element_by_class_name("video-count").text
        print("Found video, " + play_c)

  vid_count = play_c
  driver2.execute_script("window.scrollTo(0, document.body.scrollHeight);")
  time.sleep(1)
  old_url = len(url1)
old_url

36
Found video, 5404
96
126
156
186
216
246
276
306
336
366
396
426
456
474
474
done


474

In [ ]:
text = "1234hfh--cbdh---"
new = re.sub("[A-Za-z]|-","",text)
new

'1234'

In [ ]:
for i in range(0,range):
		driver2.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		time.sleep(3)


NoSuchElementException: ignored

In [ ]:
driver.back()

In [ ]:
video_url = driver.current_url
video_url = video_url.split("?")[0]
video_url = video_url.split("video/")[1]
video_url

'6896436827377519877'

In [ ]:
driver.find_elements_by_class_name("video-feed-item")[0].click()

In [ ]:
len(test['id'])

4723

In [ ]:
test = pandas.read_csv("test.csv")

In [ ]:
driver.quit()

In [ ]:

startTime = datetime.now()

print(datetime.now() - startTime)

2
0:00:00.000134


In [ ]:
from PIL import Image
import requests
from io import BytesIO
url = 'https://p16-sign-sg.tiktokcdn.com/obj/tos-alisg-p-0037/f07711bde7b64bfeaba322347da0d997_1605454266?x-expires=1606021200&x-signature=jljfnQ1lQh%2BaPSGBR1In5p1n%2BV4%3D'
response = requests.get(url)
img = Image.open(BytesIO(response.content))
width, height = img.size
height

960

In [ ]:
soup = BeautifulSoup(driver.page_source,'html.parser')
fee = soup.find('strong',{'title':'share'})
print(fee.text)

159


In [ ]:
driver.get("https://www.tiktok.com/@dina/video/6897604487549144321?lang=en")

In [ ]:
user_username = driver.find_element_by_class_name("user-username").text
video_info = driver.find_element_by_class_name("video-meta-title").text
video_likes = driver.find_element_by_class_name("like-text").text
video_comments = driver.find_element_by_class_name("comment-text").text
video_url = driver.current_url

NoSuchElementException: ignored

In [ ]:
user-nickname

⠋ TikTok Scraper Started⠙ TikTok Scraper Started⠹ TikTok Scraper Started

In [ ]:
!apt update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium bs4 pandas

Ign:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ InRelease [3,626 B]
Hit:3 http://archive.ubuntu.com/ubuntu bionic InRelease
Get:4 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Ign:5 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  Release [697 B]
Hit:7 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  Release.gpg [836 B]
Get:9 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Get:10 http://ppa.launchpad.net/c2d4u.team/c2d4u4.0+/ubuntu bionic InRelease [15.9 kB]
Get:11 http://ppa.launchpad.net/graphics-drivers/ppa/ubuntu bionic InRelease [21.3 kB]
Get:12 http://archive.ubuntu.com/ubu

In [ ]:
!apt upgrade

Reading package lists... Done
Building dependency tree       
Reading state information... Done
Calculating upgrade... Done
The following packages were automatically installed and are no longer required:
  linux-headers-4.15.0-123 linux-headers-4.15.0-123-generic
Use 'apt autoremove' to remove them.
The following NEW packages will be installed:
  linux-headers-4.15.0-124 linux-headers-4.15.0-124-generic
The following packages have been kept back:
  libcublas-dev libcublas10 libcudnn7 libcudnn7-dev libnccl-dev libnccl2
The following packages will be upgraded:
  cuda-drivers cuda-drivers-455 libc-bin libgssapi-krb5-2 libk5crypto3
  libkrb5-3 libkrb5support0 libldap-2.4-2 libldap-common libnvidia-cfg1-430
  libnvidia-cfg1-455 libnvidia-common-455 libnvidia-compute-430
  libnvidia-compute-455 libnvidia-decode-455 libnvidia-encode-455
  libnvidia-extra-455 libnvidia-fbc1-455 libnvidia-gl-430 libnvidia-gl-455
  libnvidia-ifr1-455 libpam-systemd libperl5.26 libpq5 librados2 librbd1
  libsyste

In [ ]:
from selenium import webdriver
import time
import threading
import re
from datetime import datetime
from datetime import date
import pandas
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome('chromedriver',options=options)
driver1 = webdriver.Chrome('chromedriver',options=options)
driver2 = webdriver.Chrome('chromedriver',options=options)
'''driver3 = webdriver.Chrome('chromedriver',options=options)
driver4 = webdriver.Chrome('chromedriver',options=options)
driver5 = webdriver.Chrome('chromedriver',options=options)
driver6 = webdriver.Chrome('chromedriver',options=options)
driver7 = webdriver.Chrome('chromedriver',options=options)
driver8 = webdriver.Chrome('chromedriver',options=options)'''
f = open("week1.txt","r")
file = f.readlines()
f.close()
len(file)

74

In [ ]:
def main(uri):
    kit = 0
    while kit != 1:
        try:
            driver.get(uri)
            driver.implicitly_wait(10) # seconds
            soup = BeautifulSoup(driver.page_source,'html.parser')
            head = soup.find('div',{'class':'share-title-container'})
            song_name = head.find('h1',{'class':'share-title'}).text
            song_author = head.find('h2',{'class':'share-sub-title'}).text
            videos = head.find('h2',{'class':'share-sub-title-thin'}).text
            print("Song name - %s\nSong Author - %s\nVideo count - %s" %(song_name,song_author,videos))
            if 'K' in videos:
              if '.' in videos:
                videos = videos.split('.')[0]
                video_c = videos + "000"
                print(video_c)
              else:
                videos = videos.split("K")[0]
                video_c = videos + "000"
                print(video_c)
            else:
              videos = videos.split('v')[0]
              video_c = videos.strip()
              print(video_c)
            if 'M' in videos:
              if '.' in videos:
                videos = videos.split('.')[0]
                video_c = videos + "000000"
                print(video_c)
              else:
                videos = videos.split("M")[0]
                video_c = videos + "000000"
                print(video_c)

            wait = WebDriverWait(driver2, 10)
            def send_mail(sname,fname):
                import smtplib
                from email.mime.multipart import MIMEMultipart
                from email.mime.text import MIMEText
                from email.mime.base import MIMEBase
                from email import encoders

                fromaddr = "acarsonmail@gmail.com"
                toaddr = "acarsonbot@gmail.com"

                # instance of MIMEMultipart
                msg = MIMEMultipart()

                # storing the senders email address
                msg['From'] = fromaddr

                # storing the receivers email address
                msg['To'] = toaddr

                # storing the subject
                msg['Subject'] = "Weekly Scraped Report"

                # string to store the body of the mail
                body = "Here is the csv of %s music scraped data" %(sname)

                # attach the body with the msg instance
                msg.attach(MIMEText(body, 'plain'))

                # open the file to be sent
                filename = fname
                attachment = open(filename, "rb")

                # instance of MIMEBase and named as p
                p = MIMEBase('application', 'octet-stream')

                # To change the payload into encoded form
                p.set_payload((attachment).read())

                # encode into base64
                encoders.encode_base64(p)

                p.add_header('Content-Disposition', "attachment; filename= %s" % filename)

                # attach the instance 'p' to instance 'msg'
                msg.attach(p)

                # creates SMTP session
                s = smtplib.SMTP('smtp.gmail.com', 587)

                # start TLS for security
                s.starttls()

                # Authentication
                s.login(fromaddr, "chimdindu")

                # Converts the Multipart msg into a string
                text = msg.as_string()

                # sending the mail
                s.sendmail(fromaddr, toaddr, text)
                print("sent mail")
                # terminating the session
                s.quit()

            def getvideoinfo(d,l,url,item):
              soup = BeautifulSoup(driver1.page_source,'html.parser')
              #wait.until(EC.presence_of_element_located((By.CLASS_NAME, "user-username")))
              try:
                  user_username = soup.find("h3",{"class":"author-uniqueId"}).text
              except:
                print("couldn't get username")
              soup1 = BeautifulSoup(driver.page_source,'html.parser')
              music_author = soup1.find("h2",{"class":"share-sub-title"}).text
              getvideoinfo.ma = music_author
              try:
                video_info = driver1.find_element_by_class_name("tt-video-meta-caption").text
              except:
                video_info = "no description for this user"
              video_likes = soup.find("strong",{"title":"like"}).text
              try:
                video_share = soup.find("strong",{"title":"share"}).text
              except:
                video_share = 0
              video_comments = soup.find("strong",{"title":"comment"}).text
              user_nickname,date_posted = driver1.find_element_by_class_name("author-nickname").text.split('·')
              user_nickname = user_nickname.strip()
              date_posted = date_posted.strip()
              music_id = driver.current_url
              music_id = music_id.split("?")[0]
              music_id = music_id.split("music")[1]
              music_id = re.sub("[A-Za-z]|-","",music_id)
              getvideoinfo.mi = music_id
              video_id = driver1.current_url
              video_id = video_id.split("?")[0]
              video_id = video_id.split("video/")[1]
              video_url = driver1.current_url
              print(url[item]["href"])
              nurl = url[item]["href"].split("/video")[0]
              driver2.get(nurl)
              try:
                wait.until(EC.presence_of_element_located((By.CLASS_NAME, "count-infos")))
                soup = BeautifulSoup(driver2.page_source,'html.parser')
                following = soup.find("strong",{"title":"Following"}).text
                followers = soup.find("strong",{"title":"Followers"}).text
                likes = soup.find("strong",{"title":"Likes"}).text
                profile_desc = soup.find("h2",{"class":"share-desc"}).text
                profile_url = driver2.current_url
                try:
                  is_verified = soup.find("h2",{"class":"verified"}).text
                  is_verified = "True"
                except AttributeError as e:
                  is_verified = "False"
                d["User_Followers"] = followers
                d["User_Following"] = following
                d["User_Hearts"] = likes
                d["Profile_Description"] = profile_desc
                d["Is_Verified"] = is_verified
                d["Profile_url"] = profile_url
                vid_count = 'a'
                play_c = ''
                while True:
                  soup = BeautifulSoup(driver2.page_source,'html.parser')
                  url1 = soup.find_all("a",{"class":"video-feed-item-wrapper"})
                  if vid_count == 'a':
                    for item1 in range(len(url1)):
                      if url1[item1]["href"] == url[item]["href"]:
                        play_c = driver2.find_element_by_class_name("video-count").text
                        print("Found video, " + play_c)
                        break
                  else:
                    break

                  vid_count = play_c
                  driver2.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                  time.sleep(1)

              except TimeoutException :
                print("invalid account")

              '''try:
                driver.find_element_by_class_name("close").click()
              except:
                print("already closed")'''
              #driver.get('https://www.tiktok.com/music/Because-of-You-6896223884782077953?_d=secCgsIARCbDRgBIAIoARI%2BCjx4jnuX7AefMNdouEZOo9gWbO2w5RDaJ1y7trVnIWTK0h21hJDaKqbswIqWt3g3iKHs0bHwT7K7rfEJdSAaAA%3D%3D&language=en&sec_user_id=MS4wLjABAAAAnbamQ47M0ESG8Bl6-PuMhVXof-dmWv1NXkh7NvgaQwehfBpFPY9CwbdsVVHuG43z&share_link_id=2D160053-4B13-4F16-B466-700E043190CF&share_music_id=6896223884782077953&tt_from=copy&u_code=def8cliggdfga9&user_id=6873924327805649926&utm_campaign=client_share&utm_medium=ios&utm_source=copy&source=h5_m')
              d["Username"] = user_username
              d["User_Nickname"] = user_nickname
              d["Video_ID"] = video_id
              d["Description"] = video_info
              d["Video_Hearts"] = video_likes
              d["Video_Shares"] = video_share
              d["Video_comments"] = video_comments
              d["Date_Posted"] = date_posted
              d["Video_Url"] = video_url
              d["Video plays"] = vid_count
              l.append(d)
              getvideoinfo.ci = " User's Username - %s \n Video's description - %s \n Video's Likes and Comments - %s likes  , %s comments \n Video's url - %s \n" %(user_username,video_info,video_likes,video_comments,video_url)

            def parse():
              soup = BeautifulSoup(driver.page_source,'html.parser')
              feed = soup.find_all('div',{'class':'video-feed-item'})
              print(len(feed))
              l = []
              for item in range(int(video_c)):
                try:
                    if item > 2000:
                        break
                    print("current round - " + str(item + 1))
                    d = {}
                    soup = BeautifulSoup(driver.page_source,'html.parser')
                    url = soup.find_all("a",{"class":"video-feed-item-wrapper"})
                    try:
                      element = driver.find_elements_by_class_name("video-feed-item")[item]
                    except IndexError:
                      break
                    driver.execute_script("arguments[0].scrollIntoView();", element)
                    driver1.get(url[item]["href"])
                    getvideoinfo(d,l,url,item)
                except Exception as e:
                    print(e)
                    continue
              d["Music_Author"] = getvideoinfo.ma
              d["Music_ID"] = getvideoinfo.mi
              d["Total_Videos"] = item
              today = date.today()
              today = today.strftime("%d-%m-%Y")
              csv ="%s %s.csv" % (song_name,today)
              df = pandas.DataFrame(l)
              df.to_csv(csv,mode='w',header=True)
              send_mail(song_name,csv)
              print(datetime.now() - startTime)

            startTime = datetime.now()

            parse()
            kit += 1
        except Exception as e:
            print(e)
            continue

for steps in file:
    print("song no: %s" %(str(steps)))
    main(steps)

song no: https://www.tiktok.com/music/Caroline-6897129358314440705

Song name - Caroline
Song Author - Arlo Parks
Video count - 4 videos
4
4
current round - 1
https://www.tiktok.com/@ur_moms_a_h0e_420/video/6899158972390051077
Found video, 186
current round - 2
https://www.tiktok.com/@conqueat/video/6899625471193926918
Found video, 56
current round - 3
https://www.tiktok.com/@losdemusicup/video/6900660686666419457
Found video, 36
current round - 4
https://www.tiktok.com/@aficiaoff/video/6898624727661710594
sent mail
0:00:28.472965
song no: https://www.tiktok.com/music/LUCID-6897352423573030913?_d=secCgsIARCbDRgBIAIoARI%2BCjz2IKdmKC5hC%2FsTk7l5rWnohCrchOQodqutRGVzLB9e1iBBeeviJnHOUZoLBwNkyb%2BilUSkKVCibM9RlwkaAA%3D%3D&language=en&sec_user_id=MS4wLjABAAAAnbamQ47M0ESG8Bl6-PuMhVXof-dmWv1NXkh7NvgaQwehfBpFPY9CwbdsVVHuG43z&share_link_id=1A0EBD72-6768-49A4-BE51-C8B5848B4F06&share_music_id=6897352423573030913&tt_from=copy&u_code=def8cliggdfga9&user_id=6873924327805649926&utm_campaign=client_shar

In [ ]:
driver.quit()
driver1.quit()
driver2.quit()